In [3]:
#!/usr/bin/env python3
from ncclient import manager
import sys

# Define connection parameters for your new device
device_params = {
    "host": "172.60.60.21",
    "port": 830, # Standard NETCONF over SSH port
    "username": "admin",
    "password": "admin",
    "hostkey_verify": False # Skips SSH host-key verification for first-time login
}

try:
    # 1. Establish the session. The HELLO exchange happens automatically here.
    with manager.connect(**device_params) as m:
        
        print("\n=== SUCCESS: Connected to Device ===\n")
        
        # 2. Extract and print the device capabilities directly from the manager object
        print("Device NETCONF Capabilities:")
        for capability in m.server_capabilities:
            print(f" - {capability}")
            
except Exception as e:
    print(f"Connection failed: {e}", file=sys.stderr)



=== SUCCESS: Connected to Device ===

Device NETCONF Capabilities:
 - urn:ietf:params:netconf:base:1.0
 - urn:ietf:params:netconf:base:1.1
 - urn:ietf:params:xml:ns:netconf:base:1.0
 - urn:ietf:params:xml:ns:yang:ietf-netconf-monitoring
 - urn:ietf:params:netconf:capability:writable-running:1.0
 - urn:ietf:params:netconf:capability:candidate:1.0
 - urn:ietf:params:netconf:capability:url:1.0?scheme=file,flash,ftp,http
 - http://arista.com/yang/openconfig/lldp/deviations?module=arista-lldp-deviations&revision=2022-01-17
 - http://openconfig.net/yang/aft/ni?module=openconfig-aft-network-instance&revision=2023-04-25
 - http://openconfig.net/yang/telemetry?module=openconfig-telemetry&revision=2018-11-21
 - http://openconfig.net/yang/ospfv2?module=openconfig-ospfv2&revision=2023-11-01
 - http://openconfig.net/yang/fib-types?module=openconfig-aft-types&revision=2022-05-05
 - http://arista.com/yang/openconfig/policy/deviations?module=arista-rpol-deviations&revision=2024-11-25
 - http://openco

In [1]:
"""
gNMI Connectivity & Capabilities Tester for Arista cEOS
"""
from pygnmi.client import gNMIclient
import sys

# Define your cEOS Containerlab target connection parameters
target = ('172.60.60.22', 6030)
username = 'admin'
password = 'admin'

print(f"Connecting to gNMI target {target[0]}:{target[1]}...")

try:
    # insecure=True skips TLS — correct for cEOS port 6030 in a lab
    with gNMIclient(target=target, username=username, password=password, insecure=True) as client:

        print("Connection successful. Sending Capabilities RPC...")

        capabilities = client.capabilities()

        print("\n=== Node Metadata ===")
        # pygnmi returns the key as 'gNMI_version', not 'gnmi_version'
        print(f"gNMI Version: {capabilities.get('gnmi_version', 'n/a')}")

        encodings = capabilities.get('supported_encodings', [])
        print(f"Encodings   : {', '.join(encodings) if encodings else 'n/a'}")

        print("\n=== Supported Data Models (First 5) ===")
        supported_models = capabilities.get('supported_models', [])
        for model in supported_models[:5]:
            print(f"  - {model.get('name')} (ver: {model.get('version') or 'n/a'})")

        print(f"\nTotal YANG models advertised: {len(supported_models)}")
        print("gNMI is operational on this node.")

except Exception as e:
    print(f"Failed: {e}", file=sys.stderr)
    print("Check: management api gnmi is running, port 6030 is reachable.", file=sys.stderr)
    sys.exit(1)


Connecting to gNMI target 172.60.60.22:6030...
Connection successful. Sending Capabilities RPC...

=== Node Metadata ===
gNMI Version: 0.7.0
Encodings   : json, json_ietf, ascii

=== Supported Data Models (First 5) ===
  - openconfig-ospf-policy (ver: 0.1.3)
  - arista-ale-counters-open-config-specialized-counters (ver: 1.0.0)
  - arista-telemetry-notsupported-deviations (ver: n/a)
  - openconfig-pf-srte (ver: 0.2.0)
  - arista-ntp-augments (ver: 1.0.0)

Total YANG models advertised: 232
gNMI is operational on this node.


In [5]:
import sys, os

print(sys.executable)
print(sys.path)
print(os.listdir())
print(os.getcwd())
print(os.getegid())

/home/fandre/labs/envs/automation/bin/python
['/usr/local/lib/python312.zip', '/usr/local/lib/python3.12', '/usr/local/lib/python3.12/lib-dynload', '', '/home/fandre/labs/envs/automation/lib/python3.12/site-packages']
['gnmi_capabilities.py', 'capabilities.txt', 'programmability.ipynb', '.ipynb_checkpoints']
/home/fandre/labs/ceos_vxlan_evpn/jupylabs
1000


In [4]:
from rich import print as rprint
from pygnmi.client import gNMIclient
import sys

def get_capabilities(host: str, name: str = "") -> dict | None:
    """
    Open a gNMI session to host:GNMI_PORT and issue a CapabilityRequest.
    Returns the parsed capability dict or None on failure.
    """
    label = name or host
    print(f"\n{'='*60}")
    print(f"  Node   : {label}")
    print(f"  Target : {host}:{GNMI_PORT}")
    print(f"{'='*60}")

    try:
        with gNMIclient(
            target=(host, GNMI_PORT),
            username=USERNAME,
            password=PASSWORD,
            insecure=True,          # No TLS on cEOS port 6030
        ) as gc:
            caps = gc.capabilities()

    except Exception as e:
        print(f"Failed: {e}", file=sys.stderr)
        print("Check: management api gnmi is running, port 6030 is reachable.", file=sys.stderr)
        sys.exit(1)

    return caps


# Define your cEOS Containerlab target connection parameters
TARGET = '172.60.60.22'
GNMI_PORT = 6030
USERNAME   = "admin"
PASSWORD   = "admin"

print(f"Connecting to gNMI target {TARGET}:{GNMI_PORT}...")

capabilities = get_capabilities(TARGET)
rprint(capabilities)


Connecting to gNMI target 172.60.60.22:6030...

  Node   : 172.60.60.22
  Target : 172.60.60.22:6030


{
    'supported_models': [
        {'name': 'openconfig-flexalgo', 'organization': 'OpenConfig working group', 'version': '0.1.0'},
        {
            'name': 'arista-interfaces-notsupported-deviations',
            'organization': 'Arista Networks, Inc.',
            'version': ''
        },
        {'name': 'gnsi-authz', 'organization': 'Google LLC', 'version': ''},
        {
            'name': 'arista-l1-open-config-platform-model-aug',
            'organization': 'Arista Networks <http://arista.com/>',
            'version': '1.0.0'
        },
        {'name': 'openconfig-metadata', 'organization': 'OpenConfig working group', 'version': '0.1.0'},
        {'name': 'openconfig-ospf-types', 'organization': 'OpenConfig working group', 'version': '0.1.3'},
        {
            'name': 'openconfig-aft-network-instance',
            'organization': 'OpenConfig working group',
            'version': '0.3.1'
        },
        {'name': 'openconfig-pim', 'organization': 'OpenConfig working group', 'version': '0.4.3'},
        {'name': 'arista-lldp-notsupported-deviations', 'organization': 'Arista Networks, Inc.', 'version': ''},
        {'name': 'openconfig-alarm-types', 'organization': 'OpenConfig working group', 'version': '0.2.1'},
        {
            'name': 'arista-exp-eos-qos-config',
            'organization': 'Arista Networks <http://arista.com/>',
            'version': '0.2.0'
        },
        {
            'name': 'arista-network-instance-notsupported-deviations',
            'organization': 'Arista Networks, Inc.',
            'version': ''
        },
        {
            'name': 'arista-l1-open-config-optical-channel-model-aug',
            'organization': 'Arista Networks <http://arista.com/>',
            'version': '1.0.0'
        },
        {'name': 'openconfig-platform-linecard', 'organization': 'OpenConfig working group', 'version': '1.2.0'},
        {'name': 'openconfig-platform-fabric', 'organization': 'OpenConfig working group', 'version': '0.1.0'},
        {
            'name': 'arista-mpls-augments',
            'organization': 'Arista Networks <http://arista.com/>',
            'version': '1.3.1'
        },
        {
            'name': 'arista-ale-counters-open-config-hardware-drop-counters',
            'organization': 'Arista Networks <http://arista.com/>',
            'version': '1.0.0'
        },
        {
            'name': 'arista-exp-eos-igmpsnooping',
            'organization': 'Arista Networks <http://arista.com/>',
            'version': ''
        },
        {'name': 'openconfig-segment-routing', 'organization': 'OpenConfig working group', 'version': '0.4.1'},
        {
            'name': 'arista-exp-eos-multicast',
            'organization': 'Arista Networks <http://arista.com/>',
            'version': ''
        },
        {'name': 'arista-bgp-deviations', 'organization': 'Arista Networks, Inc.', 'version': '1.27.1'},
        {'name': 'openconfig-pim-types', 'organization': 'OpenConfig working group', 'version': '0.1.1'},
        {
            'name': 'ietf-interfaces',
            'organization': 'IETF NETMOD (Network Modeling) Working Group',
            'version': ''
        },
        {'name': 'openconfig-sampling', 'organization': 'OpenConfig working group', 'version': '0.1.0'},
        {'name': 'arista-exp-eos-bgp', 'organization': 'Arista Networks, Inc.', 'version': '1.3.0'},
        {
            'name': 'openconfig-platform-integrated-circuit',
            'organization': 'OpenConfig working group',
            'version': '0.3.1'
        },
        {'name': 'openconfig-extensions', 'organization': 'OpenConfig working group', 'version': '0.5.1'},
        {'name': 'arista-telemetry-augments', 'organization': 'Arista Networks, Inc.', 'version': '1.0.0'},
        {
            'name': 'arista-l1-open-config-xcvr-oc-types',
            'organization': 'Arista Networks <http://arista.com/>',
            'version': '1.0.0'
        },
        {
            'name': 'arista-vlan